# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 refresh-priority model with public-safe, decision-support language. It reviews two research findings, compares random versus client-grouped validation, checks leakage, inspects real errors, and rewrites an over-strong claim.

## 1. Two paper findings + my methodology questions

### Finding A — Freshness Multiplier
The FlyRank paper reports that mature pages refreshed within 30 days showed a **3.2× health boost** and **57× impressions** versus the stale comparison cohort. It also warns that the very old freshness bucket is unstable because it contains very few declining pages.

**Methodology question:** How was “refreshed within 30 days” timestamped relative to the outcome window, and were refreshed pages selected because they already had stronger historical visibility or a known reason to be refreshed? If selection depended on prior performance, a simple before/after or cohort comparison can mix the refresh effect with selection and regression-to-the-mean. A stronger design would use pre-refresh performance and a clearly separated post-refresh window, ideally with a comparable untouched group.

### Finding B — AI-generated content is not penalized by default
The paper says that, in a mostly AI-authored portfolio, age-controlled model cohorts do not show a simple blanket penalty tied only to AI use.

**Methodology question:** What is the actual counterfactual for an “AI penalty” claim when almost the whole portfolio is AI-authored? Are there enough comparable non-AI pages by topic, publication period, and workflow quality? If not, the evidence supports a narrower statement about observed differences across AI workflow/model cohorts rather than a causal AI-versus-human claim.

These questions are constructive checks on label/exposure definition and whether the comparison/validation design supports the strength of the claim. The paper itself frames its work as a pattern study rather than proof of cause and effect.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
HERE=Path.cwd()
ROOT=next((p for p in [HERE,*HERE.parents,Path('/mnt/data/flyrank/repo/flyrank-ml-track-main')] if (p/'data/raw/content_refresh_anonymized.csv').exists()),None)
if ROOT is None: raise FileNotFoundError('Repository data not found. Run the notebook from the repository checkout.')
# Rebuild the processed feature vector/baseline from the committed raw data when needed.
import subprocess, sys
if not (ROOT/'data/processed/refresh_feature_vector.csv').exists(): subprocess.run([sys.executable,str(ROOT/'scripts/01_prepare_features.py')],check=True)
if not (ROOT/'data/processed/baseline_refresh_queue.csv').exists(): subprocess.run([sys.executable,str(ROOT/'scripts/02_baseline_score.py')],check=True)
raw=pd.read_csv(ROOT/'data/processed/refresh_feature_vector.csv')
print('Rows:',len(raw),'| positive rate:',round(raw.is_declining_label.mean(),3),'| clients:',raw.client_id.nunique())

Rows: 30000 | positive rate: 0.542 | clients: 32


## 2. My model under an honest split (before/after)

**Before:** stratified random row holdout. The same client can appear in train and test.

**After:** client-grouped holdout. Entire clients are held out, so the model is evaluated on clients it never saw during training. Precision@50 is the primary metric because the task is ranking a small review queue.

In [2]:
NUM=["search_volume","competition","cpc","word_count","char_count","log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d","days_with_impressions","days_with_sessions","content_age_days","days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
CAT=["competition_level","content_type","main_intent","age_tier","freshness_tier","word_count_tier","impression_tier","position_tier"]
def features(df):
    a=df[[c for c in NUM if c in df]].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0)
    b=pd.get_dummies(df[[c for c in CAT if c in df]].fillna('unknown').astype(str),prefix=CAT,dtype=float)
    return pd.concat([a.reset_index(drop=True),b.reset_index(drop=True)],axis=1)
def p50(y,s,k=50):
    order=np.argsort(-np.asarray(s))[:k]; return float(np.asarray(y)[order].mean())
def fit(train,test):
    X=features(raw); y=raw.is_declining_label.astype(int)
    m=RandomForestClassifier(class_weight='balanced_subsample',max_depth=10,min_samples_leaf=25,n_estimators=200,n_jobs=-1,random_state=42)
    m.fit(X.iloc[train],y.iloc[train]); return m,m.predict_proba(X.iloc[test])[:,1],X,y
idx=np.arange(len(raw)); y=raw.is_declining_label.astype(int)
rtr,rte=train_test_split(idx,test_size=.2,random_state=42,stratify=y)
clients=raw.client_id.fillna('unknown').astype(str); u=clients.drop_duplicates().to_numpy(); rng=np.random.default_rng(42); test_clients=set(rng.permutation(u)[:round(len(u)*.2)]); mask=clients.isin(test_clients).to_numpy(); gtr=idx[~mask]; gte=idx[mask]
rm,rs,X,y=fit(rtr,rte); gm,gs,_,_=fit(gtr,gte)
b=pd.read_csv(ROOT/'data/processed/baseline_refresh_queue.csv').set_index('content_id').baseline_refresh_score
bs=raw.iloc[gte].content_id.map(b).fillna(0).to_numpy()
rows=[
 ['Random row (before)',len(rtr),len(rte),y.iloc[rte].mean(),roc_auc_score(y.iloc[rte],rs),average_precision_score(y.iloc[rte],rs),p50(y.iloc[rte],rs)],
 ['Client grouped (after)',len(gtr),len(gte),y.iloc[gte].mean(),roc_auc_score(y.iloc[gte],gs),average_precision_score(y.iloc[gte],gs),p50(y.iloc[gte],gs)],
 ['Week-4 baseline on grouped test','-',len(gte),y.iloc[gte].mean(),np.nan,np.nan,p50(y.iloc[gte],bs)]
]
comparison=pd.DataFrame(rows,columns=['split','train_rows','test_rows','base_rate','roc_auc','avg_precision','precision_at_50'])
print(comparison.round(3).to_string(index=False))
print('Random/client overlap:',len(set(clients.iloc[rtr])&set(clients.iloc[rte])),'| Grouped overlap:',len(set(clients.iloc[gtr])&set(clients.iloc[gte])))

                          split train_rows  test_rows  base_rate  roc_auc  avg_precision  precision_at_50
            Random row (before)      24000       6000      0.542    0.758          0.768             0.90
         Client grouped (after)      27675       2325      0.391    0.750          0.618             0.74
Week-4 baseline on grouped test          -       2325      0.391      NaN            NaN             0.24
Random/client overlap: 31 | Grouped overlap: 0


### Before/after interpretation

The random-row score is not the deployment claim because it allows within-client overlap. The grouped score answers the harder question: can the model rank decline-labelled pages for clients it never saw? The **0.160 Precision@50 gap** is itself a validation finding. I use the grouped result for decision-support.

## 3. Leakage audit

The leakage hunt follows three checks: label-derived features, future/overlapping windows, and decision-derived product flags. `is_declining_label` is derived from `trend_direction`/`trend_pct`, so those fields are excluded. `client_id` and `content_id` are grouping/joining identifiers only. The Week-4 `baseline_refresh_score` is a comparator, not a model input. Trailing-90-day performance fields are treated as snapshot features; this starter dataset does not prove a future-time deployment result.

In [3]:
label_cols=[c for c in raw.columns if c in {'is_declining_label','trend_direction','trend_pct'}]
feature_leaks=[c for c in X.columns if c in label_cols or c in {'client_id','content_id'}]
print('Label/outcome-derived columns:',label_cols)
print('Excluded IDs:',[c for c in ['client_id','content_id'] if c in raw.columns])
print('Actual leakage matches in model matrix:',feature_leaks)
print('Feature count:',X.shape[1])
print('Baseline score is kept outside the model matrix: True')
print('Timeline caveat: snapshot/trailing-90d features are not a future-time evaluation.')

Label/outcome-derived columns: ['trend_direction', 'trend_pct', 'is_declining_label']
Excluded IDs: ['client_id', 'content_id']
Actual leakage matches in model matrix: []
Feature count: 52
Baseline score is kept outside the model matrix: True
Timeline caveat: snapshot/trailing-90d features are not a future-time evaluation.


## 4. Claim rewrite

### Earlier claim (too strong)
> “Random Forest identifies the pages that will decline and improves refresh decisions by 208%.”

### Safer claim
> **Observed:** On this 30,000-row starter dataset, the Random Forest measured higher Precision@50 than the Week-4 baseline. **Measured:** Precision@50 was **0.740** on the client-grouped test set versus **0.240** for the Week-4 baseline on the same grouped rows. **Directional:** the grouped result is lower than the **0.900** measured under a random row split, showing that validation design materially affects the estimate. **Decision-support:** the score can prioritize human review of possible decline candidates; it is not evidence that a page will causally decline or that refreshing it will improve traffic.

In [4]:
# Three real grouped-holdout failures, shown without IDs, clients, URLs, or queries.
t=raw.iloc[gte].copy(); t['prob']=gs; t['pred']=(gs>=.5).astype(int); t['actual']=y.iloc[gte].to_numpy(); t['error']=np.where((t.actual==0)&(t.pred==1),'false_positive',np.where((t.actual==1)&(t.pred==0),'false_negative','correct')); t['distance']=(t.prob-.5).abs()
e=t[t.error!='correct'].sort_values('distance').head(3)
cols=['error','prob','actual','trend_direction','impressions_90d','avg_position','ctr','content_age_days','days_since_last_update','word_count']
print(e[cols].round(3).to_string(index=False))

         error  prob  actual trend_direction  impressions_90d  avg_position   ctr  content_age_days  days_since_last_update  word_count
false_negative 0.500       1            down            22617           7.1  0.21               140                       8      3037.0
false_negative 0.499       1            down               27          11.5  0.00               333                      20      1988.0
false_positive 0.501       0          stable               42          10.9 19.05               287                      20       889.0


### Failure interpretation

The examples show both false negatives and false positives near the decision boundary. That supports using the model as a **review priority**, not as an automatic publishing, removal, or traffic-guarantee decision.

## Self-check

- [x] Two paper findings and constructive methodology questions
- [x] Label/comparison design questioned for both findings
- [x] Random-row and client-grouped validation compared
- [x] Base rates printed beside metrics
- [x] Grouped split has zero train/test client overlap
- [x] Label-derived and decision-derived fields excluded
- [x] Failure examples inspected without client names, URLs, or queries
- [x] Claims use observed, measured, directional, decision-support language
- [x] Fixed random seed 42; notebook executed top to bottom and rebuilds processed inputs from committed raw data when needed
- [x] Ready for `work/notebooks/w06_validation_audit.ipynb`